# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gaurav443201/flyrank-ml-internship-work/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I am working on the Refresh / Content Opportunity Scoring lane.

This is primarily a ranking/scoring task. The goal is to rank pages by their likelihood of being useful candidates for content refresh review. A ranking is more useful than a simple yes/no classification because the content team has limited time and needs to know which pages should be reviewed first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is whether a page is showing a declining trend. I use the observed trend_direction value as a proxy for refresh opportunity: pages with trend_direction equal to "down" are treated as declining pages.

This is an observed outcome in the dataset. It does not mean that refreshing a page will definitely improve its performance. The model is intended to identify pages that deserve review based on signals that were available before the outcome.

In [4]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "/content/flyrank-ml-internship-starter"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

print("Repository ready:", REPO_DIR)

Repository ready: /content/flyrank-ml-internship-starter


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My main success metric is Precision@50.

Precision@50 measures the percentage of the top 50 pages ranked by the model that are actually declining. A higher Precision@50 means that more of the pages sent to the top of the review queue are genuinely declining.

This metric fits the refresh opportunity task because a content team has limited time and needs useful pages near the top of the review list. I will compare the model with a simple hand-written baseline using the same evaluation setup.

In [5]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Metric: Precision@50")
print("Higher Precision@50 means more declining pages appear in the top 50.")

Metric: Precision@50
Higher Precision@50 means more declining pages appear in the top 50.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one page observation.

Each row represents one anonymized page and its observed search and content performance. The dataframe contains page-level signals such as impressions, average position, CTR, content age, word count, and trend direction.

The model will rank individual pages for content refresh review rather than ranking clients or individual search queries.

In [7]:
import pandas as pd
import numpy as np

CSV_PATH = "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(CSV_PATH)

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Data loaded successfully")
print("Rows:", len(df))
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Data loaded successfully
Rows: 30000
Declining rate: 0.542


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as "old pages with high impressions should be refreshed" is useful as a transparent baseline, but the relationship between content age, impressions, position, CTR, and declining trends can involve several signals at the same time.

A model can combine these signals and learn different combinations instead of relying on one or two manually selected thresholds. I will therefore compare the learned approach with a simple hand-written rule rather than assuming that ML is automatically better.

The final output is decision-support: it helps prioritize pages for human review. It does not automatically decide what content should be changed.

In [8]:
from sklearn.tree import DecisionTreeClassifier, export_text

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining_label"]

tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree.fit(X, y)

print("Learned decision tree:")
print(export_text(tree, feature_names=features))

Learned decision tree:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.